# <font color="darkblue">**Descarga automatizada de Diarios de Sesiones del Congreso**</font>


---

**Autoría:** Sara Sampayo Sande  

**Institución:** Universidad Miguel Hernández de Elche  
**Fecha:** 1/03/2026  
**Contacto:** ssampayo@crimina.es  
**DOI / Versión:** [Identificador o versión]  

---

## 📌 **Descripción de la herramienta**

Esta herramienta, implementada en **Python** y diseñada para ejecutarse en **Google Colab**, permite la descarga automatizada y filtrada de los **Diarios de Sesiones del Congreso de los Diputados** a partir de palabras clave.

El programa simula una búsqueda avanzada en el portal oficial de publicaciones, extrayendo los resultados directamente desde la **API interna del sitio web**. Para ello, envía una petición `POST` con los parámetros seleccionados por la persona usuaria —**legislatura**, **tipo de publicación** (Diarios de Sesiones, Boletines Oficiales o todos), **cámara** (Congreso, Senado, Cortes Generales o todas) y una o varias **palabras clave** separadas por punto y coma— y recupera un listado estructurado en formato **JSON** que contiene los metadatos de cada documento, incluyendo su **CVE (Código de Validación Electrónica)** y la **URL directa al archivo PDF**.

A continuación, el script descarga automáticamente todos los PDFs identificados, organizándolos en carpetas separadas por palabra clave y legislatura, e incorpora pausas configurables para no sobrecargar el servidor. Una funcionalidad adicional permite generar, para cada combinación de búsqueda, un **enlace directo al portal del Congreso** con los filtros aplicados, facilitando la supervisión visual de los resultados.

Toda la configuración se realiza mediante una **interfaz intuitiva de formularios en Colab**, lo que hace que la herramienta sea accesible para personas sin conocimientos de programación, especialmente investigadoras e investigadores en ciencias sociales, políticas y jurídicas.

---

## 🎯 **Objetivo de la investigación**

Esta herramienta ha sido desarrollada en el marco de una **tesis doctoral** orientada al análisis del discurso parlamentario en torno a las políticas de igualdad y violencia de género. Su objetivo es facilitar la construcción de **corpus textuales sistemáticos y replicables** a partir de fuentes oficiales, garantizando la trazabilidad de los datos y la transparencia del proceso.

---

## 📚 **Uso en publicaciones**

Si esta herramienta es utilizada en una publicación académica, se ruega citarla del siguiente modo:

> [Apellido, Nombre] ([Año]). *Herramienta de descarga de Diarios de Sesiones del Congreso*. Disponible en: [URL del repositorio o cuaderno]. Versión [Nº de versión].

---

## ⚠️ **Nota importante**

La búsqueda por texto libre solo está disponible a partir de la **VI Legislatura**. En legislaturas anteriores este filtro no se encuentra operativo.

---
<font color="gray"><i>Última actualización: 9/03/2026</i></font>

# 🔍 Cómo usar el buscador

Esta herramienta permite descargar automáticamente los Diarios de Sesiones del Congreso filtrados por palabras clave.

---

## ⚙️ **Configuración inicial**

En la **primera celda** (configuración), completa los siguientes campos:

| Campo | Formato | Ejemplo |
|-------|---------|---------|
| **Palabras clave** | Separadas por `;` | `Libertad Sexual; Violencia de género` |
| **Legislaturas** | Números separados por `,` | `15,14,13,12` |
| **Tipo de publicación** | Desplegable | `Diarios de Sesiones` / `Boletines Oficiales` / `Todos` |
| **Cámara** | Desplegable | `Congreso` / `Senado` / `Cortes Xerais` / `Todas` |
| **Carpeta de salida** | Texto libre | `descargas_congreso` |

---

## 🚀 **Ejecución**

1. Ejecuta las celdas en orden:
   - **Celda 1**: Instalación (solo una vez)
   - **Celda 2**: Configuración (rellenar y ejecutar)
   - **Celda 3**: Funciones auxiliares (no tocar)
   - **Celda 4**: Búsqueda y descarga (ejecutar y esperar)

2. El programa mostrará en tiempo real:
   - 📄 Progreso por página
   - 📊 Número de documentos encontrados
   - ✅ Confirmación de descargas
   - 🔗 Enlace de supervisión (para comprobar resultados en el navegador)

---

## 📁 **Resultados**

Cada carpeta contiene:
- Los archivos PDF (formato: `DSCD-XX-PL-XXX.PDF`)
- Un archivo `cves.txt` con la lista de CVEs descargados

---

## ⚠️ **Nota importante**

El campo de **búsqueda por texto libre solo está disponible a partir de la VI Legislatura**. En legislaturas anteriores este filtro no funciona.

---

## ❓ **¿Problemas?**

- Si ves errores `403`, ejecuta la **Celda 6** para renovar las cookies
- Ajusta las pausas en "Opciones avanzadas" si el servidor va lento
- Revisa el enlace de supervisión para verificar los resultados manualmente

In [ ]:
# @title 1. 📦 INSTALACIÓN Y CONFIGURACIÓN INICIAL
!pip install requests beautifulsoup4 tqdm -q

import requests
import os
import time
import json
from tqdm import tqdm
from urllib.parse import quote

print("✅ Instalación completada")

# Obtener cookies frescas automáticamente
print("\n🍪 Obteniendo cookies del Congreso...")

try:
    # Hacer una petición inicial para obtener cookies
    session = requests.Session()
    response = session.get('https://www.congreso.es/gl/busqueda-de-publicaciones', timeout=10)

    # Extraer cookies
    cookies = session.cookies.get_dict()

    # Guardar JSESSIONID si existe
    JSESSIONID = cookies.get('JSESSIONID', '')

    if JSESSIONID:
        print(f"✅ Cookies obtenidas correctamente")
        print(f"   JSESSIONID: {JSESSIONID[:30]}...")
    else:
        print("⚠️ No se obtuvo JSESSIONID, usando cookie por defecto")
        JSESSIONID = 'esRur9kVmDnbOYAqryVEbekB9nENIQPNV48PDFDn.cgdpjbnode2pro'

except Exception as e:
    print(f"⚠️ Error obteniendo cookies: {e}")
    print("   Usando cookie por defecto")
    JSESSIONID = 'esRur9kVmDnbOYAqryVEbekB9nENIQPNV48PDFDn.cgdpjbnode2pro'

# Guardar la cookie para las demás celdas
COOKIES = {'JSESSIONID': JSESSIONID}
print("\n✅ Listo para continuar")

✅ Instalación completada

🍪 Obteniendo cookies del Congreso...
⚠️ No se obtuvo JSESSIONID, usando cookie por defecto

✅ Listo para continuar


In [ ]:
# @title ⚙️ <b><font color="darkblue" size="+2"> 2. CONFIGURACIÓN DE BÚSQUEDA</font></b>

# ======================================================================
# 🎯 PALABRAS CLAVE - Separa cada una con punto y coma (;)
# ======================================================================
#@markdown <font color="blue"><b>📝 Introduce las palabras clave separadas por ;</b></font>
PALABRAS_CLAVE = "Violencia de género"  # @param {type:"string"}

# ======================================================================
# 📅 LEGISLATURAS - Escribe los números separados por comas (ej: 15,14,13)
# ======================================================================
#@markdown <font color="blue"><b>📌 Legislaturas (números separados por comas):</b></font>
LEGISLATURAS_INPUT = "10, 11"  # @param {type:"string"}

# ======================================================================
# 📄 TIPO DE PUBLICACIÓN - ¿Qué quieres buscar?
# ======================================================================
#@markdown <font color="blue"><b>📄 Tipo de publicación:</b></font>
TIPO_PUBLICACION = "Diarios de Sesiones"  # @param ["Diarios de Sesiones", "Boletines Oficiales", "Todos"]

# Mapeo de tipos a códigos
TIPO_CODIGO = {
    "Diarios de Sesiones": "D",
    "Boletines Oficiales": "B",
    "Todos": ""
}

# ======================================================================
# 🏛️ CÁMARA / SECCIÓN - ¿Qué cámara te interesa?
# ======================================================================
#@markdown <font color="blue"><b>🏛️ Cámara / Sección:</b></font>
CAMARA = "Congreso de los Diputados"  # @param ["Congreso de los Diputados", "Senado", "Cortes Xerais", "Todas"]

# Mapeo de cámaras a códigos
CAMARA_CODIGO = {
    "Congreso de los Diputados": "CONGRESO",
    "Senado": "SENADO",
    "Cortes Xerais": "CORTES",
    "Todas": ""
}

# ======================================================================
# 📁 CARPETA DE SALIDA
# ======================================================================
#@markdown <font color="blue"><b>📁 Carpeta donde se guardarán los PDFs:</b></font>
CARPETA_BASE = "descargas_congreso"  # @param {type:"string"}

# ======================================================================
# 🔧 CONFIGURACIÓN AVANZADA (desplegar si necesitas ajustar)
# ======================================================================
#@markdown ---
#@markdown <font color="purple"><b>🔧 OPCIONES AVANZADAS (clic para desplegar)</b></font>
MODO_DEBUG = False  # @param {type:"boolean"}
PAUSA_ENTRE_DESCARGAS = 0.5  # @param {type:"slider", min:0.1, max:2.0, step:0.1}
PAUSA_ENTRE_PAGINAS = 1.0    # @param {type:"slider", min:0.5, max:3.0, step:0.5}

# ======================================================================
# 🔄 PROCESAR CONFIGURACIÓN (NO TOCAR)
# ======================================================================
import os

# Procesar palabras clave (separar por ;)
PALABRAS_CLAVE = [p.strip() for p in PALABRAS_CLAVE.split(';') if p.strip()]

# Procesar legislaturas (separar por comas)
LEGISLATURAS = []
for leg in LEGISLATURAS_INPUT.split(','):
    try:
        LEGISLATURAS.append(int(leg.strip()))
    except:
        pass

# Obtener códigos de tipo y cámara
TIPO_SELECCIONADO = TIPO_CODIGO[TIPO_PUBLICACION]
CAMARA_SELECCIONADA = CAMARA_CODIGO[CAMARA]

# ======================================================================
# 📊 MOSTRAR RESUMEN
# ======================================================================
print("=" * 70)
print("✅ CONFIGURACIÓN CARGADA")
print("=" * 70)
print(f"📌 PALABRAS CLAVE ({len(PALABRAS_CLAVE)}):")
for i, p in enumerate(PALABRAS_CLAVE, 1):
    print(f"   {i}. {p}")
print(f"\n📅 LEGISLATURAS: {LEGISLATURAS}")
print(f"\n📄 TIPO: {TIPO_PUBLICACION} (código: '{TIPO_SELECCIONADO}')")
print(f"🏛️ CÁMARA: {CAMARA} (código: '{CAMARA_SELECCIONADA}')")
print(f"\n📁 CARPETA: {CARPETA_BASE}")
if MODO_DEBUG:
    print(f"\n⚙️ PAUSAS: {PAUSA_ENTRE_DESCARGAS}s | {PAUSA_ENTRE_PAGINAS}s (modo debug)")
print("=" * 70)

# Guardar variable silenciosamente para otras celdas
_ = %store CARPETA_BASE

✅ CONFIGURACIÓN CARGADA
📌 PALABRAS CLAVE (1):
   1. Violencia de género

📅 LEGISLATURAS: [10, 11]

📄 TIPO: Diarios de Sesiones (código: 'D')
🏛️ CÁMARA: Congreso de los Diputados (código: 'CONGRESO')

📁 CARPETA: descargas_congreso
Stored 'CARPETA_BASE' (str)


In [ ]:
# @title 3. 🛠️ FUNCIONES AUXILIARES (NO MODIFICAR)
import re
import os
import time
import requests
from urllib.parse import quote

# Cabeceras para simular un navegador
HEADERS = {
    'accept': 'application/json, text/javascript, */*; q=0.01',
    'content-type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'x-requested-with': 'XMLHttpRequest',
}

# Cookies (se obtienen en la Celda 1)
try:
    COOKIES
except NameError:
    COOKIES = {'JSESSIONID': 'esRur9kVmDnbOYAqryVEbekB9nENIQPNV48PDFDn.cgdpjbnode2pro'}

def buscar_en_legislatura(legislatura, palabra_clave, pagina=1):
    """
    Busca documentos en una legislatura específica usando la API interna.
    Ahora acepta número de página para navegar correctamente.
    """
    url = "https://www.congreso.es/gl/busqueda-de-publicaciones"

    params = {
        'p_p_id': 'publicaciones',
        'p_p_lifecycle': '2',
        'p_p_resource_id': 'filtrarListado',
    }

    data = {
        '_publicaciones_legislatura': str(legislatura),
        '_publicaciones_texto': f'"{palabra_clave}"',
        '_publicaciones_tipoBusqueda': '0',
        '_publicaciones_publicacion': TIPO_SELECCIONADO,
        '_publicaciones_seccion': CAMARA_SELECCIONADA,
        '_publicaciones_paginaActual': str(pagina),  # ¡AHORA SÍ USA LA PÁGINA!
    }

    # Eliminar campos vacíos
    data = {k: v for k, v in data.items() if v != ''}

    try:
        respuesta = requests.post(url, params=params, data=data, headers=HEADERS, cookies=COOKIES, timeout=15)
        respuesta.raise_for_status()
        return respuesta.json()
    except Exception as e:
        print(f"   ❌ Error en búsqueda (página {pagina}): {e}")
        return None

def extraer_documentos(data):
    """
    Extrae la lista de documentos de la respuesta JSON de la API.
    """
    docs = []
    if isinstance(data, dict):
        for key, value in data.items():
            if key.startswith('documento') and isinstance(value, dict):
                if 'diario' in value and 'cve' in value:
                    docs.append({
                        'cve': value['cve'],
                        'url': f"https://www.congreso.es{value['diario']}",
                        'orga': value.get('orga', ''),
                        'fecha': value.get('fecha', '')
                    })
    return docs

def obtener_total_paginas(data):
    """
    Calcula el número total de páginas a partir de los resultados.
    """
    if isinstance(data, dict):
        if 'publicaciones_encontradas' in data:
            try:
                total = int(data['publicaciones_encontradas'])
                if total > 0:
                    return (total + 19) // 20
            except:
                pass
    return 1

def descargar_pdf(url, ruta):
    """
    Descarga un archivo PDF desde una URL y lo guarda en la ruta especificada.
    Incluye verificación de que la descarga fue exitosa.
    Retorna True si la descarga fue exitosa, False en caso contrario.
    """
    try:
        # Crear el directorio si no existe
        os.makedirs(os.path.dirname(ruta), exist_ok=True)

        # Mostrar información de depuración (opcional)
        # print(f"      🌐 Descargando: {url}")

        # Realizar la descarga
        headers = {'User-Agent': 'Mozilla/5.0'}
        respuesta = requests.get(url, headers=headers, stream=True, timeout=30)
        respuesta.raise_for_status()

        # Verificar que el contenido sea un PDF
        content_type = respuesta.headers.get('content-type', '').lower()
        if 'application/pdf' not in content_type and not url.lower().endswith('.pdf'):
            print(f"      ⚠️ Advertencia: Puede no ser un PDF. Tipo: {content_type}")

        # Guardar el archivo
        with open(ruta, 'wb') as f:
            for chunk in respuesta.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        # Verificar que el archivo se descargó correctamente
        if os.path.exists(ruta) and os.path.getsize(ruta) > 1000:  # Mínimo 1KB
            tamaño_kb = os.path.getsize(ruta) / 1024
            print(f"      ✅ Descargado: {os.path.basename(ruta)} ({tamaño_kb:.1f} KB)")
            return True
        else:
            print(f"      ❌ Archivo demasiado pequeño o vacío: {ruta}")
            if os.path.exists(ruta):
                os.remove(ruta)
            return False

    except Exception as e:
        print(f"      ❌ Error en descarga: {e}")
        return False

def generar_enlace_supervision(legislatura, palabra):
    """
    Genera un enlace para supervisar la búsqueda manualmente en el navegador.
    """
    palabra_codificada = quote(palabra)

    enlace = (f"https://www.congreso.es/gl/busqueda-de-publicaciones"
              f"?p_p_id=publicaciones"
              f"&p_p_lifecycle=0"
              f"&_publicaciones_legislatura={legislatura}"
              f"&_publicaciones_texto=%22{palabra_codificada}%22"
              f"&_publicaciones_publicacion={TIPO_SELECCIONADO}"
              f"&_publicaciones_seccion={CAMARA_SELECCIONADA}")

    return enlace

def descargar_pdf_silencioso(url, ruta):
    """Descarga un PDF sin imprimir nada (solo éxito/fracaso silencioso)."""
    try:
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        headers = {'User-Agent': 'Mozilla/5.0'}
        resp = requests.get(url, headers=headers, stream=True, timeout=30)
        resp.raise_for_status()

        with open(ruta, 'wb') as f:
            for chunk in resp.iter_content(8192):
                f.write(chunk)
        return True
    except:
        return False

print("✅ Funciones auxiliares cargadas correctamente")
print("   • buscar_en_legislatura(): OK")
print("   • extraer_documentos(): OK")
print("   • obtener_total_paginas(): OK")
print("   • descargar_pdf(): OK (con verificación real)")
print("   • generar_enlace_supervision(): OK")

✅ Funciones auxiliares cargadas correctamente
   • buscar_en_legislatura(): OK
   • extraer_documentos(): OK
   • obtener_total_paginas(): OK
   • descargar_pdf(): OK (con verificación real)
   • generar_enlace_supervision(): OK


In [ ]:
# @title 4. 🔍 EJECUTAR BÚSQUEDA Y DESCARGA (OUTPUT LIMPIO)
import time
from urllib.parse import quote
from tqdm import tqdm

# Crear carpeta base
os.makedirs(CARPETA_BASE, exist_ok=True)

# Estadísticas globales
total_global = 0
resultados_por_legislatura = {}

print("=" * 70)
print("🚀 INICIANDO BÚSQUEDA AUTOMÁTICA")
print("=" * 70)

for palabra in PALABRAS_CLAVE:
    print(f"\n📌 BUSCANDO: '{palabra}'")

    for legislatura in LEGISLATURAS:
        print(f"\n   📄 Legislatura {legislatura}")

        # Enlace
        enlace = generar_enlace_supervision(legislatura, palabra)
        print(f"      🔗 Enlace: {enlace}")

        # Carpeta
        carpeta = f"{CARPETA_BASE}/{palabra.replace(' ', '_')}/leg_{legislatura}"
        os.makedirs(carpeta, exist_ok=True)

        # Obtener primera página
        data = buscar_en_legislatura(legislatura, palabra, pagina=1)
        if not data:
            print("      ❌ Error")
            continue

        if 'publicaciones_encontradas' in data:
            total_docs = int(data['publicaciones_encontradas'])
            print(f"      📊 Documentos encontrados: {total_docs}")
            if total_docs == 0:
                continue

        total_paginas = obtener_total_paginas(data)

        # Recopilar documentos
        todos_docs = []
        for pagina in range(1, total_paginas + 1):
            if pagina > 1:
                data = buscar_en_legislatura(legislatura, palabra, pagina=pagina)
            if data:
                docs = extraer_documentos(data)
                todos_docs.extend(docs)

        print(f"      📋 Total en lista: {len(todos_docs)} documentos")

        # Descargar con barra de progreso (UNA SOLA LÍNEA)
        descargados = 0
        with tqdm(total=len(todos_docs), desc=f"      Descargando", unit="PDF", leave=True) as pbar:
            for idx, doc in enumerate(todos_docs, 1):
                nombre = f"{idx:04d}_{doc['cve']}.PDF"
                ruta = os.path.join(carpeta, nombre)

                if descargar_pdf_silencioso(doc['url'], ruta):
                    descargados += 1
                pbar.update(1)
                time.sleep(PAUSA_ENTRE_DESCARGAS)

        print(f"      ✅ {descargados}/{len(todos_docs)} descargados")

        # Guardar lista
        with open(os.path.join(carpeta, 'lista_completa.txt'), 'w') as f:
            for idx, doc in enumerate(todos_docs, 1):
                f.write(f"{idx:04d}\t{doc['cve']}\n")

        resultados_por_legislatura[(palabra, legislatura)] = len(todos_docs)
        total_global += len(todos_docs)

print("\n" + "=" * 70)
print("📊 RESUMEN GLOBAL")
print("=" * 70)
for (palabra, leg), count in resultados_por_legislatura.items():
    print(f"📌 '{palabra}' - Legislatura {leg}: {count} documentos")
print(f"\n✅ TOTAL GLOBAL: {total_global} documentos")
print(f"📁 Carpeta base: {CARPETA_BASE}")
print("=" * 70)

el de arriba se supone que es el  final

In [ ]:
# @title 4. 🔍 EJECUTAR BÚSQUEDA Y DESCARGA (CON PAGINACIÓN CORREGIDA)
import time
from urllib.parse import quote
from tqdm import tqdm

# Crear carpeta base
os.makedirs(CARPETA_BASE, exist_ok=True)

# Estadísticas globales
total_global = 0
resultados_por_legislatura = {}

print("=" * 70)
print("🚀 INICIANDO BÚSQUEDA AUTOMÁTICA (PAGINACIÓN CORREGIDA)")
print("=" * 70)

for palabra in PALABRAS_CLAVE:
    print(f"\n📌 BUSCANDO: '{palabra}'")

    for legislatura in LEGISLATURAS:
        print(f"\n   📄 Legislatura {legislatura}")

        # 🔗 GENERAR ENLACE
        enlace = generar_enlace_supervision(legislatura, palabra)
        print(f"      🔗 Enlace: {enlace}")

        # Carpeta
        carpeta = f"{CARPETA_BASE}/{palabra.replace(' ', '_')}/leg_{legislatura}"
        os.makedirs(carpeta, exist_ok=True)

        # Obtener primera página para saber total
        data = buscar_en_legislatura(legislatura, palabra, pagina=1)
        if not data:
            print("      ❌ Error en búsqueda")
            continue

        if 'publicaciones_encontradas' in data:
            total_docs = int(data['publicaciones_encontradas'])
            print(f"      📊 Documentos encontrados: {total_docs}")
            if total_docs == 0:
                continue

        total_paginas = obtener_total_paginas(data)
        print(f"      📄 Total páginas: {total_paginas}")

        # ===== PARTE CORREGIDA: RECOPILAR TODAS LAS PÁGINAS =====
        todos_docs = []
        cves_unicos = set()

        # Página 1
        print(f"      📄 Procesando página 1/{total_paginas}...")
        docs_pag1 = extraer_documentos(data)
        todos_docs.extend(docs_pag1)
        for doc in docs_pag1:
            cves_unicos.add(doc['cve'])

        # Páginas 2 en adelante
        for pagina in range(2, total_paginas + 1):
            print(f"      📄 Procesando página {pagina}/{total_paginas}...")
            data_pag = buscar_en_legislatura(legislatura, palabra, pagina=pagina)
            if data_pag:
                docs = extraer_documentos(data_pag)
                todos_docs.extend(docs)
                for doc in docs:
                    cves_unicos.add(doc['cve'])
            time.sleep(PAUSA_ENTRE_PAGINAS)

        print(f"      ✅ Total documentos en lista: {len(todos_docs)}")
        print(f"      ✅ CVEs únicos encontrados: {len(cves_unicos)}")
        # ===== FIN PARTE CORREGIDA =====

        # DESCARGAR (con numeración para mantener orden)
        print(f"\n      ⬇️ DESCARGANDO {len(todos_docs)} DOCUMENTOS...")
        print("      " + "-" * 40)

        descargados = 0
        errores = 0

        for idx, doc in enumerate(todos_docs, 1):
            nombre = f"{idx:04d}_{doc['cve']}.PDF"
            ruta = os.path.join(carpeta, nombre)

            print(f"         [{idx:3d}/{len(todos_docs):3d}] {doc['cve']}... ", end="")

            if descargar_pdf_silencioso(doc['url'], ruta):
                descargados += 1
                print("✅")
            else:
                errores += 1
                print("❌")

            time.sleep(PAUSA_ENTRE_DESCARGAS)

        print(f"      " + "-" * 40)
        print(f"      ✅ Descargas: {descargados} correctas, {errores} errores")

        # Guardar lista completa
        lista_path = os.path.join(carpeta, 'lista_completa.txt')
        with open(lista_path, 'w', encoding='utf-8') as f:
            for idx, doc in enumerate(todos_docs, 1):
                f.write(f"{idx:04d}\t{doc['cve']}\t{doc.get('orga', '')}\n")

        # Guardar lista de CVEs únicos
        unicos_path = os.path.join(carpeta, 'cves_unicos.txt')
        with open(unicos_path, 'w', encoding='utf-8') as f:
            for cve in sorted(cves_unicos):
                f.write(f"{cve}\n")

        print(f"      💾 Listas guardadas")

        # Actualizar estadísticas
        resultados_por_legislatura[(palabra, legislatura)] = {
            'total': len(todos_docs),
            'unicos': len(cves_unicos)
        }
        total_global += len(todos_docs)

print("\n" + "=" * 70)
print("📊 RESUMEN GLOBAL")
print("=" * 70)
for (palabra, leg), stats in resultados_por_legislatura.items():
    print(f"📌 '{palabra}' - Legislatura {leg}: {stats['total']} docs ({stats['unicos']} únicos)")
print(f"\n✅ TOTAL GLOBAL: {total_global} documentos descargados")
print(f"📁 Carpeta base: {CARPETA_BASE}")
print("=" * 70)

🚀 INICIANDO BÚSQUEDA AUTOMÁTICA (PAGINACIÓN CORREGIDA)

📌 BUSCANDO: 'Violencia de género'

   📄 Legislatura 10
      🔗 Enlace: https://www.congreso.es/gl/busqueda-de-publicaciones?p_p_id=publicaciones&p_p_lifecycle=0&_publicaciones_legislatura=10&_publicaciones_texto=%22Violencia%20de%20g%C3%A9nero%22&_publicaciones_publicacion=D&_publicaciones_seccion=CONGRESO
      📊 Documentos encontrados: 271
      📄 Total páginas: 14
      📄 Procesando página 1/14...
      📄 Procesando página 2/14...
      📄 Procesando página 3/14...
      📄 Procesando página 4/14...
      📄 Procesando página 5/14...
      📄 Procesando página 6/14...
      📄 Procesando página 7/14...
      📄 Procesando página 8/14...
      📄 Procesando página 9/14...
      📄 Procesando página 10/14...
      📄 Procesando página 11/14...
      📄 Procesando página 12/14...
      📄 Procesando página 13/14...
      📄 Procesando página 14/14...
      ✅ Total documentos en lista: 231
      ✅ CVEs únicos encontrados: 231

      ⬇️ DESCARG

In [ ]:
# @title 📊 5. RESUMEN DE DESCARGAS REALIZADAS
import os
import pandas as pd
from tabulate import tabulate

# Recuperar CARPETA_BASE de la celda de configuración
try:
    %store -r CARPETA_BASE
    print(f"📁 Usando carpeta: {CARPETA_BASE}")
except:
    print("⚠️ No se pudo recuperar CARPETA_BASE, usa valor por defecto")
    CARPETA_BASE = "descargas_congreso"

print("=" * 70)
print("📊 RESUMEN GLOBAL DE DESCARGAS")
print("=" * 70)

# Verificar que la carpeta existe
if not os.path.exists(CARPETA_BASE):
    print(f"❌ La carpeta '{CARPETA_BASE}' no existe.")
    print("   Asegúrate de haber ejecutado la celda de descargas primero.")
else:
    # Estructura para guardar el resumen
    resumen = []

    # Recorrer la carpeta base
    for palabra in os.listdir(CARPETA_BASE):
        ruta_palabra = os.path.join(CARPETA_BASE, palabra)
        if not os.path.isdir(ruta_palabra):
            continue

        # Recorrer legislaturas dentro de cada palabra
        for leg_folder in os.listdir(ruta_palabra):
            ruta_leg = os.path.join(ruta_palabra, leg_folder)
            if not os.path.isdir(ruta_leg):
                continue

            # Extraer número de legislatura del nombre de carpeta
            try:
                legislatura = leg_folder.replace('leg_', '')
            except:
                legislatura = leg_folder

            # Contar archivos PDF
            pdfs = [f for f in os.listdir(ruta_leg) if f.upper().endswith('.PDF')]
            num_pdfs = len(pdfs)

            if num_pdfs > 0:
                resumen.append({
                    'Palabra clave': palabra.replace('_', ' '),
                    'Legislatura': legislatura,
                    'PDFs descargados': num_pdfs,
                    'Carpeta': f"{palabra}/{leg_folder}"
                })

    # Crear DataFrame para mejor visualización
    if resumen:
        df_resumen = pd.DataFrame(resumen)

        # Totales por palabra clave
        print("\n📌 TOTALES POR PALABRA CLAVE:")
        print("-" * 50)
        for palabra in df_resumen['Palabra clave'].unique():
            total_palabra = df_resumen[df_resumen['Palabra clave'] == palabra]['PDFs descargados'].sum()
            print(f"   • '{palabra}': {total_palabra} PDFs")

        # Totales por legislatura
        print("\n📌 TOTALES POR LEGISLATURA:")
        print("-" * 50)
        for leg in sorted(df_resumen['Legislatura'].unique()):
            total_leg = df_resumen[df_resumen['Legislatura'] == leg]['PDFs descargados'].sum()
            print(f"   • Legislatura {leg}: {total_leg} PDFs")

        # Resumen detallado en tabla
        print("\n📋 DETALLE POR PALABRA Y LEGISLATURA:")
        print("-" * 50)
        df_resumen = df_resumen.sort_values(['Palabra clave', 'Legislatura'])
        print(tabulate(df_resumen[['Palabra clave', 'Legislatura', 'PDFs descargados']],
                       headers='keys', tablefmt='grid', showindex=False))

        # Total global
        total_global = df_resumen['PDFs descargados'].sum()
        print("\n" + "=" * 50)
        print(f"✅ TOTAL GLOBAL: {total_global} PDFs descargados")
        print(f"📁 Carpeta base: {CARPETA_BASE}")

        # Guardar resumen como CSV
        csv_path = f"/content/resumen_descargas.csv"
        df_resumen.to_csv(csv_path, index=False)
        print(f"💾 Resumen guardado como: 'resumen_descargas.csv'")

    else:
        print("❌ No se encontraron descargas en la carpeta especificada.")

print("=" * 70)

📁 Usando carpeta: descargas_congreso
📊 RESUMEN GLOBAL DE DESCARGAS

📌 TOTALES POR PALABRA CLAVE:
--------------------------------------------------
   • 'Violencia de género': 973 PDFs

📌 TOTALES POR LEGISLATURA:
--------------------------------------------------
   • Legislatura 10: 231 PDFs
   • Legislatura 11: 22 PDFs
   • Legislatura 14: 460 PDFs
   • Legislatura 15: 260 PDFs

📋 DETALLE POR PALABRA Y LEGISLATURA:
--------------------------------------------------
+---------------------+---------------+--------------------+
| Palabra clave       |   Legislatura |   PDFs descargados |
+=====================+===============+====================+
| Violencia de género |            10 |                231 |
+---------------------+---------------+--------------------+
| Violencia de género |            11 |                 22 |
+---------------------+---------------+--------------------+
| Violencia de género |            14 |                460 |
+---------------------+---------------+-

In [ ]:
# @title 🔍 6. DETECCIÓN Y ELIMINACIÓN DE DUPLICADOS
import os
from collections import defaultdict
import pandas as pd
from tabulate import tabulate

print("=" * 70)
print("🔍 DETECCIÓN Y ELIMINACIÓN DE DUPLICADOS")
print("=" * 70)
print(f"📁 Analizando: {CARPETA_BASE}")

# @markdown ---
MODO = "Eliminar duplicados"  # @param ["Solo listar", "Eliminar duplicados"]
# @markdown ---

# Recopilar archivos
archivos = []
for root, dirs, files in os.walk(CARPETA_BASE):
    for file in files:
        if file.upper().endswith('.PDF'):
            partes = root.split(os.sep)
            palabra = partes[1] if len(partes) > 1 else "desconocida"
            legislatura = "desconocida"
            for p in partes:
                if p.startswith('leg_'):
                    legislatura = p.replace('leg_', '')
                    break

            if '_' in file:
                cve = file.split('_', 1)[1].replace('.PDF', '')
                indice = file.split('_', 1)[0]
            else:
                cve = file.replace('.PDF', '')
                indice = "0000"

            archivos.append({
                'leg': legislatura,
                'cve': cve,
                'archivo': file,
                'indice': indice,
                'ruta': os.path.join(root, file),
                'tamaño_kb': os.path.getsize(os.path.join(root, file)) / 1024
            })

total_inicial = len(archivos)
print(f"📊 Total archivos: {total_inicial}")

# Estadísticas iniciales rápidas
leg_counts = {}
for leg in sorted(set(a['leg'] for a in archivos)):
    leg_counts[leg] = len([a for a in archivos if a['leg'] == leg])

# Procesar eliminación
total_eliminados = 0
espacio_liberado = 0

for leg in sorted(leg_counts.keys()):
    archivos_leg = [a for a in archivos if a['leg'] == leg]

    # Agrupar por CVE
    cve_dict = {}
    for a in archivos_leg:
        if a['cve'] not in cve_dict:
            cve_dict[a['cve']] = []
        cve_dict[a['cve']].append(a)

    # Eliminar duplicados
    for cve, lista in cve_dict.items():
        if len(lista) > 1:
            # Ordenar por índice
            lista_ordenada = sorted(lista, key=lambda x: x['indice'])
            # Conservar el primero, eliminar los demás
            for a in lista_ordenada[1:]:
                if MODO == "Eliminar duplicados":
                    try:
                        os.remove(a['ruta'])
                        total_eliminados += 1
                        espacio_liberado += a['tamaño_kb'] / 1024
                    except:
                        pass

# RESULTADO FINAL
print("\n" + "=" * 70)
print("📊 RESUMEN FINAL")
print("=" * 70)

if MODO == "Solo listar":
    print("\n🔍 MODO SOLO LISTAR - No se eliminó nada")
    print(f"   Archivos analizados: {total_inicial}")
else:
    print(f"\n✅ ELIMINACIÓN COMPLETADA")
    print(f"   • Archivos eliminados: {total_eliminados}")
    print(f"   • Archivos conservados: {total_inicial - total_eliminados}")
    print(f"   • Espacio liberado: {espacio_liberado:.2f} MB")

    # Contar archivos finales
    archivos_finales = 0
    leg_final = defaultdict(int)
    for root, dirs, files in os.walk(CARPETA_BASE):
        for file in files:
            if file.upper().endswith('.PDF'):
                archivos_finales += 1
                for p in root.split(os.sep):
                    if p.startswith('leg_'):
                        leg_final[p.replace('leg_', '')] += 1
                        break

    print(f"\n📁 Total final: {archivos_finales} archivos")
    for leg in sorted(leg_final.keys()):
        print(f"   • Legislatura {leg}: {leg_final[leg]} archivos")

print("\n" + "=" * 70)
print("✅ Proceso completado")

🔍 DETECCIÓN Y ELIMINACIÓN DE DUPLICADOS
📁 Analizando: descargas_congreso
📊 Total archivos: 293

📊 RESUMEN FINAL

✅ ELIMINACIÓN COMPLETADA
   • Archivos eliminados: 0
   • Archivos conservados: 293
   • Espacio liberado: 0.00 MB

📁 Total final: 293 archivos
   • Legislatura 10: 231 archivos
   • Legislatura 11: 22 archivos
   • Legislatura 14: 20 archivos
   • Legislatura 15: 20 archivos

✅ Proceso completado


In [ ]:
# @title 🔍 7. ANÁLISIS DE DUPLICADOS ENTRE LEGISLATURAS
import os
from collections import defaultdict
import pandas as pd
from tabulate import tabulate

print("=" * 70)
print("🔍 ANÁLISIS DE DUPLICADOS ENTRE LEGISLATURAS")
print("=" * 70)
print(f"📁 Analizando: {CARPETA_BASE}")
print("-" * 70)

# Recopilar todos los archivos con su CVE y legislatura
archivos = []
for root, dirs, files in os.walk(CARPETA_BASE):
    for file in files:
        if file.upper().endswith('.PDF'):
            # Extraer legislatura
            legislatura = "desconocida"
            for p in root.split(os.sep):
                if p.startswith('leg_'):
                    legislatura = p.replace('leg_', '')
                    break

            # Extraer CVE (sin prefijo numérico)
            if '_' in file:
                cve = file.split('_', 1)[1].replace('.PDF', '')
            else:
                cve = file.replace('.PDF', '')

            archivos.append({
                'legislatura': legislatura,
                'cve': cve,
                'archivo': file,
                'ruta': os.path.join(root, file)
            })

print(f"\n📊 Total archivos analizados: {len(archivos)}")

# Crear DataFrame para análisis
df = pd.DataFrame(archivos)

# 1. CVEs que aparecen en múltiples legislaturas
print("\n" + "=" * 70)
print("📌 CVEs QUE APARECEN EN MÚLTIPLES LEGISLATURAS")
print("=" * 70)

# Agrupar por CVE y contar legislaturas únicas
cve_por_leg = df.groupby('cve')['legislatura'].nunique().reset_index()
cve_por_leg.columns = ['cve', 'num_legislaturas']
cve_por_leg = cve_por_leg.sort_values('num_legislaturas', ascending=False)

cves_repetidos = cve_por_leg[cve_por_leg['num_legislaturas'] > 1]

if len(cves_repetidos) > 0:
    print(f"\n⚠️ Se encontraron {len(cves_repetidos)} CVEs que aparecen en varias legislaturas:")

    # Mostrar tabla con los más repetidos
    tabla = []
    for _, row in cves_repetidos.head(15).iterrows():
        cve = row['cve']
        num_leg = row['num_legislaturas']
        # Obtener lista de legislaturas donde aparece
        legislaturas = sorted(df[df['cve'] == cve]['legislatura'].unique())
        archivos_ejemplo = df[df['cve'] == cve].iloc[0]['archivo']
        tabla.append([cve, num_leg, ', '.join(legislaturas), archivos_ejemplo])

    print(tabulate(tabla, headers=['CVE', 'Legislaturas', 'Años', 'Ejemplo'], tablefmt='grid'))

    if len(cves_repetidos) > 15:
        print(f"\n... y {len(cves_repetidos) - 15} CVEs más")
else:
    print("\n✅ No hay CVEs repetidos entre legislaturas")

# 2. Resumen por CVE
print("\n" + "=" * 70)
print("📊 RESUMEN POR CVE")
print("=" * 70)

# Estadísticas generales
total_cves_unicos = df['cve'].nunique()
print(f"\n📊 Estadísticas:")
print(f"   • Total CVEs únicos: {total_cves_unicos}")
print(f"   • Total archivos: {len(df)}")
print(f"   • Ratio archivos/CVE: {len(df)/total_cves_unicos:.2f}")

# Distribución de CVEs por número de legislaturas
print(f"\n📊 Distribución de CVEs por número de legislaturas:")
distribucion = cve_por_leg['num_legislaturas'].value_counts().sort_index()
for num_leg, count in distribucion.items():
    print(f"   • {num_leg} legislatura{'s' if num_leg > 1 else ''}: {count} CVEs")

# 3. Mostrar ejemplos concretos de CVEs repetidos
if len(cves_repetidos) > 0:
    print("\n" + "=" * 70)
    print("📋 EJEMPLOS DETALLADOS DE CVEs REPETIDOS")
    print("=" * 70)

    # Tomar los 3 primeros CVEs con más repeticiones
    for _, row in cves_repetidos.head(3).iterrows():
        cve = row['cve']
        print(f"\n📄 CVE: {cve}")
        print("-" * 40)

        # Mostrar todos los archivos de este CVE ordenados por legislatura
        archivos_cve = df[df['cve'] == cve].sort_values('legislatura')
        for _, arch in archivos_cve.iterrows():
            print(f"   • Legislatura {arch['legislatura']}: {arch['archivo']}")

# 4. Conclusión
print("\n" + "=" * 70)
print("📌 CONCLUSIÓN")
print("=" * 70)

if len(cves_repetidos) > 0:
    print(f"\n⚠️ Hay {len(cves_repetidos)} CVEs que se repiten entre diferentes legislaturas.")
    print("   Esto significa que algunos nombres de archivo son idénticos en distintos años.")
    print("   IMPORTANTE: Son documentos DIFERENTES aunque tengan el mismo nombre.")
    print("\n   Recomendación: Mantener los archivos separados por carpetas de legislatura")
    print("   para no mezclar documentos de diferentes años.")
else:
    print("\n✅ No hay CVEs repetidos entre legislaturas.")
    print("   Todos los nombres de archivo son únicos a través de los años.")

print("\n" + "=" * 70)
print("✅ Análisis completado")

🔍 ANÁLISIS DE DUPLICADOS ENTRE LEGISLATURAS
📁 Analizando: descargas_congreso
----------------------------------------------------------------------

📊 Total archivos analizados: 293

📌 CVEs QUE APARECEN EN MÚLTIPLES LEGISLATURAS

✅ No hay CVEs repetidos entre legislaturas

📊 RESUMEN POR CVE

📊 Estadísticas:
   • Total CVEs únicos: 293
   • Total archivos: 293
   • Ratio archivos/CVE: 1.00

📊 Distribución de CVEs por número de legislaturas:
   • 1 legislatura: 293 CVEs

📌 CONCLUSIÓN

✅ No hay CVEs repetidos entre legislaturas.
   Todos los nombres de archivo son únicos a través de los años.

✅ Análisis completado


In [ ]:
# @title 📦 8. DESCARGAR TODO EL CORPUS COMO ZIP
import os
import zipfile
from datetime import datetime

# Forzar importación directa de google.colab
import google.colab
from google.colab import files

print("=" * 70)
print("📦 COMPRIMIR Y DESCARGAR CORPUS COMPLETO")
print("=" * 70)

# @markdown ### ⚙️ Opciones de descarga
INCLUIR_FECHA = True  # @param {type:"boolean"}
ELIMINAR_DESPUES = False  # @param {type:"boolean"}

# Crear nombre del archivo
if INCLUIR_FECHA:
    fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_zip = f"corpus_congreso_{fecha_actual}.zip"
else:
    nombre_zip = "corpus_congreso.zip"

ruta_zip = f"/content/{nombre_zip}"

print(f"\n📁 Carpeta a comprimir: {CARPETA_BASE}")
print(f"📦 Archivo ZIP: {nombre_zip}")
print("-" * 70)

# Verificar que la carpeta existe
if not os.path.exists(CARPETA_BASE):
    print(f"❌ Error: La carpeta '{CARPETA_BASE}' no existe")
else:
    # Contar archivos primero
    total_archivos = 0
    total_tamaño = 0
    archivos_por_leg = {}

    for root, dirs, files in os.walk(CARPETA_BASE):
        for file in files:
            if file.upper().endswith('.PDF'):
                total_archivos += 1
                file_path = os.path.join(root, file)
                total_tamaño += os.path.getsize(file_path)

                # Extraer legislatura
                for p in root.split(os.sep):
                    if p.startswith('leg_'):
                        leg = p.replace('leg_', '')
                        archivos_por_leg[leg] = archivos_por_leg.get(leg, 0) + 1
                        break

    print(f"\n📊 ESTADÍSTICAS DEL CORPUS:")
    print(f"   • Total archivos: {total_archivos}")
    print(f"   • Tamaño total: {total_tamaño / (1024*1024):.2f} MB")
    print(f"\n   📌 Distribución por legislatura:")
    for leg in sorted(archivos_por_leg.keys()):
        print(f"      • Legislatura {leg}: {archivos_por_leg[leg]} archivos")

    print("\n" + "-" * 70)
    print("⏳ Comprimiendo archivos...")

    # Crear ZIP
    archivos_comprimidos = 0
    with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(CARPETA_BASE):
            for file in files:
                if file.upper().endswith('.PDF'):
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, start=os.path.dirname(CARPETA_BASE))
                    zipf.write(file_path, arcname)
                    archivos_comprimidos += 1

                    if archivos_comprimidos % 50 == 0:
                        print(f"      📍 Progreso: {archivos_comprimidos}/{total_archivos} archivos")

    print(f"      ✅ Completado: {archivos_comprimidos}/{total_archivos} archivos")

    # Tamaño del ZIP
    tamaño_zip = os.path.getsize(ruta_zip) / (1024 * 1024)
    print(f"\n📊 ZIP creado: {tamaño_zip:.2f} MB")

    # Descargar
    print("\n⬇️ INICIANDO DESCARGA...")

    # Alternativa si files.download falla
    try:
        files.download(ruta_zip)
        print("✅ Descarga completada")
    except Exception as e:
        print(f"⚠️ Error con descarga automática: {e}")
        print("\n📌 DESCARGA MANUAL:")
        print(f"   1. Ve al panel izquierdo de Colab (ícono de carpeta)")
        print(f"   2. Busca el archivo: {nombre_zip}")
        print(f"   3. Haz clic derecho y selecciona 'Descargar'")

    # Limpiar si se solicita
    if ELIMINAR_DESPUES:
        os.remove(ruta_zip)
        print("🧹 Archivo ZIP eliminado del servidor")

print("\n" + "=" * 70)
print("✅ PROCESO COMPLETADO")
print("=" * 70)

📦 COMPRIMIR Y DESCARGAR CORPUS COMPLETO

📁 Carpeta a comprimir: descargas_congreso
📦 Archivo ZIP: corpus_congreso_20260309_125009.zip
----------------------------------------------------------------------

📊 ESTADÍSTICAS DEL CORPUS:
   • Total archivos: 293
   • Tamaño total: 198.86 MB

   📌 Distribución por legislatura:
      • Legislatura 10: 231 archivos
      • Legislatura 11: 22 archivos
      • Legislatura 14: 20 archivos
      • Legislatura 15: 20 archivos

----------------------------------------------------------------------
⏳ Comprimiendo archivos...
      📍 Progreso: 50/293 archivos
      📍 Progreso: 100/293 archivos
      📍 Progreso: 150/293 archivos
      📍 Progreso: 200/293 archivos
      📍 Progreso: 250/293 archivos
      ✅ Completado: 293/293 archivos

📊 ZIP creado: 114.53 MB

⬇️ INICIANDO DESCARGA...
⚠️ Error con descarga automática: 'list' object has no attribute 'download'

📌 DESCARGA MANUAL:
   1. Ve al panel izquierdo de Colab (ícono de carpeta)
   2. Busca el arch